# Data Structures from Scratch — Meta Interview

You already have the **patterns** (`educative/`) and the **timed drills** (`blind/`).
This notebook is the layer underneath: *what the structure actually is*, why its
complexity is what it is, and the one trap Meta interviewers use for each.

**How to use it.** Run every cell top to bottom — each section is
`teach → build it yourself → use the built-in → one real problem`.
The from-scratch implementations are not for the interview (use the built-ins);
they are so the complexity table stops being memorised and starts being *obvious*.

Section 14 is a **self-test with no answers** — same spirit as
`blind/practice-without-view.ipynb`.

---

## The decision table — symptom to structure

This is the table that wins interviews. The interviewer describes a symptom;
you name the structure in 5 seconds.

| The question says... | Reach for | Why |
|---|---|---|
| "how many times does each…", "seen before?", "duplicate" | `dict` / `set` | O(1) membership |
| "top K", "K largest/smallest", "K closest" | heap of size K | O(n log K) |
| "median of a stream" | two heaps | max-heap low half, min-heap high half |
| "matching brackets", "undo", "next greater element" | stack | LIFO / monotonic |
| "shortest path, all edges cost 1" | BFS + `deque` | first visit = shortest |
| "all paths", "does a path exist", "flood fill" | DFS (recursion or stack) | explores one branch fully |
| "prerequisites", "build order", "no cycles" | topological sort | Kahn's algorithm |
| "are these two in the same group", "merge groups" | union-find | near O(1) merge + query |
| "prefix", "autocomplete", "starts with" | trie | O(len) not O(n·len) |
| "sorted array" + "find" | binary search | O(log n) |
| "sorted array" + "pair/triplet sums" | two pointers | O(n), no extra space |
| "contiguous subarray/substring" | sliding window | O(n) |
| "count the ways", "min/max cost to reach" | DP table | overlapping subproblems |
| "in place", "O(1) extra space" | two pointers / index math | forbids the hash map |
| "merge K sorted things" | heap | O(n log K) |
| "cycle in a linked list", "find the middle" | fast/slow pointers | O(1) space |

## Where each structure lives in your repo

| Section here | Your existing practice |
|---|---|
| 3 Hash map | `educative/ Hash Maps`, `blind/2_most_common_comment.py` |
| 4 Stack | `educative/Stacks` |
| 5 Deque | `educative/3_Sliding_window` |
| 6 Linked list | `educative/5_In_Place_Manipulation_Linked_List`, `2_fast_slow` |
| 7 Heap | `educative/8_Top_K_Elements`, `6_Two_Heaps`, `7_K_way_Merge` |
| 8 Tree | `educative/20_Tree Depth_First Search`, `21_Tree_Breadth_First_Search` |
| 10 Graph | `educative/19_Graphs` |
| 12 DP | `educative/13_Dynamic_Programming` |

In [ ]:
# Run this first. Everything below depends on it.
from collections import Counter, defaultdict, deque, namedtuple
import heapq, time, sys, random

random.seed(42)          # deterministic, so reruns match
print(sys.version.split()[0])

---
# 1. Dynamic array — Python `list`

A `list` is a **contiguous block of pointers** plus two numbers: `len` (used) and
`allocated` (capacity). That single fact explains the whole complexity table.

- `append` — write at index `len`, bump the counter. O(1).
  When `len == allocated`, CPython allocates a bigger block (~1.125×) and copies
  everything. That copy is O(n), but it happens rarely enough that the cost
  *spread over all appends* is O(1). This is **amortised** O(1) — say that word.
- `lst[i]` — one pointer arithmetic step. O(1). This is what a linked list cannot do.
- `insert(0, x)` / `pop(0)` — every later element shifts one slot. **O(n)**.
  A loop doing `pop(0)` is O(n²) and it is the single most common accidental
  quadratic in interview code. Use `deque` (section 5).
- `x in lst` — linear scan. **O(n)**. Use a `set` if you do it in a loop.

| op | cost |
|---|---|
| `lst[i]`, `lst[i] = v`, `len` | O(1) |
| `append`, `pop()` (from end) | O(1) amortised |
| `insert(0,x)`, `pop(0)`, `remove(x)`, `del lst[0]` | O(n) |
| `x in lst` | O(n) |
| `sort()` | O(n log n) |
| slice `lst[a:b]` | O(b−a) — it **copies** |

**The trap:** an interviewer asks for O(1) extra space, and your solution slices.
`lst[1:]` is a new list — that is O(n) space, and inside a loop it is O(n²) time.
Pass indices instead of slicing.

In [ ]:
# --- Proof that growth is geometric, not one-at-a-time ---------------------
lst, seen = [], []
for i in range(1000):
    lst.append(i)
    cap = (sys.getsizeof(lst) - sys.getsizeof([])) // 8   # 8 bytes per pointer
    if not seen or cap != seen[-1]:
        seen.append(cap)
print("capacities as the list grows:", seen[:14], "...")
# Each jump is ~1.125x the last -> O(1) amortised append.

# --- The O(n) vs O(1) end of the list -------------------------------------
N = 50_000
t = time.perf_counter(); a = []
for i in range(N): a.append(i)            # O(1) each
fast = time.perf_counter() - t

t = time.perf_counter(); b = []
for i in range(N): b.insert(0, i)         # O(n) each -> O(n^2) total
slow = time.perf_counter() - t
print(f"append {fast*1000:7.1f} ms | insert(0) {slow*1000:7.1f} ms | {slow/fast:.0f}x slower")

# --- Slicing copies; indices do not --------------------------------------
big = list(range(100_000))
assert sys.getsizeof(big[1:]) > 100_000     # a whole new list
assert big[1:][0] == big[1]                 # same value, O(n) to get it
print("\nslicing allocates -- pass (lo, hi) indices to stay O(1) space")

In [ ]:
# THE array technique: two pointers, in place, O(1) extra space.
def remove_duplicates_sorted(nums):
    """Compact a sorted list in place; return the new logical length.

    write = the boundary of the kept region. Read forward, only copy back
    when the value is new. Classic 'slow/fast index' array move.
    """
    if not nums:
        return 0
    write = 1
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:
            nums[write] = nums[read]
            write += 1
    return write

a = [1, 1, 2, 2, 2, 3, 4, 4]
k = remove_duplicates_sorted(a)
assert a[:k] == [1, 2, 3, 4], a[:k]

def reverse_in_place(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        nums[lo], nums[hi] = nums[hi], nums[lo]   # tuple swap, no temp var
        lo, hi = lo + 1, hi - 1
    return nums

assert reverse_in_place([1, 2, 3, 4, 5]) == [5, 4, 3, 2, 1]
assert reverse_in_place([]) == []
print("two-pointer array drills pass")

---
# 2. Strings — immutable, and why that bites

A Python `str` is an **immutable** array of characters. Every "modification"
builds a brand-new string and copies.

So `s += c` in a loop is O(len(s)) per step → **O(n²) total**.
Build a `list` of pieces and `"".join(pieces)` at the end → O(n).

> CPython has an unofficial in-place optimisation for `s += c` when the string has
> exactly one reference, so you may not always *see* the quadratic. Never rely on
> it — it silently vanishes the moment anything else holds a reference, and an
> interviewer reading your code will mark `+=` in a loop as O(n²) regardless.

Other things worth knowing cold:

- `s[::-1]` reverses — O(n), idiomatic, fine to use.
- `sorted(s)` returns a **list** of chars; `"".join(sorted(s))` is the anagram key.
- `s.split()` with no argument splits on any run of whitespace and drops empties —
  that is what you want for word-count questions; `s.split(" ")` keeps empties.
- Comparing strings is O(len), not O(1). A `dict` with long string keys still
  hashes the whole key.
- For a fixed lowercase alphabet, a 26-slot list beats a dict (constant factor).

In [ ]:
# --- += vs join ----------------------------------------------------------
N = 30_000
t = time.perf_counter()
s = ""
for i in range(N):
    s = s + "x"          # deliberately not '+=', to defeat CPython's in-place trick
concat = time.perf_counter() - t

t = time.perf_counter()
parts = []
for i in range(N):
    parts.append("x")
joined = "".join(parts)
join_t = time.perf_counter() - t

assert s == joined
print(f"s = s + c {concat*1000:7.1f} ms | ''.join {join_t*1000:7.1f} ms -> build a list, join once")

# --- the idioms you must not have to think about --------------------------
assert "hello"[::-1] == "olleh"
assert "".join(sorted("listen")) == "".join(sorted("silent"))     # anagram key
assert "  a  b ".split() == ["a", "b"]                            # drops empties
assert "a,,b".split(",") == ["a", "", "b"]                        # keeps them
assert "Hello World".lower().split() == ["hello", "world"]
print("string idioms pass")

In [ ]:
# Meta-flavoured: 'most mentioned word' -- exactly your blind/7 problem,
# written the way it should be written.
def most_mentioned_word(comments):
    """Most frequent word across comments; ties broken alphabetically.

    Ties MUST be broken deterministically or the interviewer asks 'what if two
    words tie?' and your answer is 'uh, whichever'. min() on (-count, word)
    gives highest count, then lexicographically smallest.
    """
    counts = Counter()
    for c in comments:
        counts.update(c.lower().split())          # update() folds in a whole list
    if not counts:
        return None
    return min(counts, key=lambda w: (-counts[w], w))

assert most_mentioned_word(["love this", "Love it", "hate love"]) == "love"   # 3x
assert most_mentioned_word(["love this", "Love it", "hate it"]) == "it"
#       ^ love:2 and it:2 tie, so the alphabetical rule picks 'it'. Write the
#         tie case as a TEST, not a hope -- this is exactly where people get caught.
assert most_mentioned_word(["a b", "b a"]) == "a"        # 2-2 tie -> alphabetical
assert most_mentioned_word([]) is None
assert most_mentioned_word(["", "  "]) is None           # whitespace only
print("most_mentioned_word passes, ties included")

---
# 3. Hash map & hash set — `dict`, `set`

**The most important structure in the entire interview.** If you only truly
master one thing, master this.

### How it works
An array of buckets. `hash(key) % n_buckets` picks a bucket. Two keys can land in
the same bucket — a **collision**. CPython resolves collisions by **open
addressing** (probe to another slot); the textbook alternative is **chaining**
(a list per bucket), which is what we implement below because it is easier to see.
When the table gets ~2/3 full it **resizes** and rehashes everything.

### Complexity — and the honest answer
- Average: `d[k]`, `d[k]=v`, `del d[k]`, `k in d` → **O(1)**
- Worst case: **O(n)** — if every key collides. Say this out loud; it shows you
  know it is a hash table, not magic.
- Iterating: O(n), and **in insertion order** (guaranteed since Python 3.7).

### Keys must be hashable
Immutable things are: `int`, `str`, `tuple`, `frozenset`.
`list`, `dict`, `set` are **not** — `d[[1,2]]` raises `TypeError`.
Need a list as a key? `tuple(lst)`. Need a set as a key? `frozenset(s)`.

### The three tools — know which one to reach for
| Tool | Use it when |
|---|---|
| `Counter(iterable)` | counting. `.most_common(k)` is a heap internally |
| `defaultdict(list)` | grouping — no `if k not in d` boilerplate |
| `dict.get(k, default)` | reading a maybe-missing key without mutating |
| `dict.setdefault(k, [])` | grouping in one line when you can't import |

**The trap:** `defaultdict` *creates* the key when you merely read it.
`len(dd)` after `if dd[k]` is bigger than you expect. Use `.get()` to peek.

In [ ]:
class HashMap:
    """Teaching implementation: buckets + chaining. Shows WHY it's O(1) average."""

    def __init__(self, n_buckets=8):
        self._buckets = [[] for _ in range(n_buckets)]
        self._size = 0

    def _bucket(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def put(self, key, value):
        b = self._bucket(key)
        for i, (k, _) in enumerate(b):
            if k == key:                 # found: overwrite, size unchanged
                b[i] = (key, value)
                return
        b.append((key, value))
        self._size += 1
        if self._size > len(self._buckets) * 2 // 3:      # load factor 0.66
            self._resize()

    def get(self, key, default=None):
        for k, v in self._bucket(key):   # scan ONE bucket, not the whole table
            if k == key:
                return v
        return default

    def _resize(self):
        old = self._buckets
        self._buckets = [[] for _ in range(len(old) * 2)]
        self._size = 0
        for b in old:
            for k, v in b:               # rehash: bucket index depends on n_buckets
                self.put(k, v)

    def __len__(self):
        return self._size

    def load_profile(self):
        return [len(b) for b in self._buckets]


m = HashMap()
for i, w in enumerate("the quick brown fox jumps over the lazy dog".split()):
    m.put(w, i)
assert len(m) == 8                        # 'the' appeared twice -> overwritten
assert m.get("fox") == 3
assert m.get("cat") is None
assert m.get("cat", -1) == -1
print("buckets:", m.load_profile(), "<- short chains = O(1) average lookup")

In [ ]:
# --- the built-ins, used correctly ---------------------------------------
words = "the quick brown fox jumps over the lazy dog the end".split()

c = Counter(words)
assert c["the"] == 3 and c["missing"] == 0        # Counter never KeyErrors
assert c.most_common(1) == [("the", 3)]

groups = defaultdict(list)                        # grouping without boilerplate
for w in words:
    groups[len(w)].append(w)
assert groups[3] == ["the", "fox", "the", "dog", "the", "end"]   # insertion order

# THE defaultdict trap:
dd = defaultdict(list)
_ = dd["never_assigned"]                          # a read... that writes
assert "never_assigned" in dd and len(dd) == 1
plain = {}
assert plain.get("never_assigned") is None and len(plain) == 0   # .get() is safe
print("Counter / defaultdict / .get() semantics pass")

# --- hashability ----------------------------------------------------------
d = {}
d[(1, 2)] = "tuple ok"
d[frozenset({1, 2})] = "frozenset ok"
try:
    d[[1, 2]] = "nope"
except TypeError as e:
    print("list as key ->", e)

In [ ]:
# Two canonical hash-map problems. Both are 'nested loop -> hash map' rewrites.

def two_sum(nums, target):
    """Indices of the pair summing to target. O(n) time, O(n) space.

    The move: instead of searching for the complement, REMEMBER what you've seen.
    """
    seen = {}                                # value -> index
    for i, v in enumerate(nums):
        if target - v in seen:
            return [seen[target - v], i]
        seen[v] = i
    return []

assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 3], 6) == [0, 1]          # duplicate values
assert two_sum([1, 2], 99) == []

def group_anagrams(words):
    """Group words that are anagrams. O(n · k log k) for k = word length.

    The move: design a KEY such that equal keys mean 'same group'.
    Sorted letters is that key. (Even faster: a 26-tuple of counts -> O(n·k).)
    """
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

got = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(map(sorted, got)) == sorted(map(sorted,
    [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]))
print("two_sum + group_anagrams pass")

---
# 4. Stack — LIFO

Just a `list`: `append` to push, `pop()` to pop. Both O(1). **Never use
`pop(0)`** — that is a queue, and on a list it is O(n).

Reach for a stack when the problem has **nesting** or **"the most recent
unmatched thing"**: brackets, expression evaluation, undo, DFS without recursion.

### Monotonic stack — the one that separates candidates
A stack you keep **sorted** by popping anything that breaks the order before you
push. It answers *"for each element, what is the next greater/smaller one?"* in
**O(n) total** — every element is pushed once and popped once, even though there
is a `while` inside the `for`. Being able to justify that O(n) is the whole point.

**Recursion is a stack.** Python's default limit is ~1000 frames. On a degenerate
(linked-list-shaped) tree of 10⁵ nodes, recursive DFS blows the stack — rewrite it
iteratively with an explicit stack. Mentioning this unprompted is a strong signal.

In [ ]:
def is_balanced(s):
    """Valid brackets. The textbook stack problem."""
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:   # unmatched or mismatched
                return False
    return not stack                                    # leftovers = unclosed

assert is_balanced("({[]})")
assert not is_balanced("(]")
assert not is_balanced("((")          # unclosed
assert not is_balanced("))")          # closes nothing
assert is_balanced("")
print("is_balanced passes")

In [ ]:
def next_greater(nums):
    """For each element, the next strictly greater element to its right (-1 if none).

    O(n): the stack holds INDICES whose answer is still unknown, kept decreasing.
    Each index is pushed once and popped once -> the inner while is O(1) amortised.
    """
    out = [-1] * len(nums)
    stack = []                                   # indices, values decreasing
    for i, v in enumerate(nums):
        while stack and nums[stack[-1]] < v:     # v resolves everything smaller
            out[stack.pop()] = v
        stack.append(i)
    return out                                   # whatever is left stays -1

assert next_greater([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1]
assert next_greater([5, 4, 3]) == [-1, -1, -1]   # already decreasing
assert next_greater([1, 2, 3]) == [2, 3, -1]     # already increasing
assert next_greater([]) == []

def daily_temperatures(temps):
    """Same machine, different output: how many DAYS until warmer."""
    out = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            out[j] = i - j                       # distance, not value
        stack.append(i)
    return out

assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
print("monotonic stack passes -- one machine, two problems")

---
# 5. Queue & deque — FIFO

A queue needs `pop` from the **front**, which a `list` does in O(n).
`collections.deque` is a **doubly linked list of fixed-size blocks**, so both ends
are O(1).

| op | `deque` | `list` |
|---|---|---|
| `append` / `pop` (right) | O(1) | O(1) amortised |
| `appendleft` / `popleft` | **O(1)** | **O(n)** |
| `dq[i]` random access | **O(n)** | **O(1)** |

That last row is the tradeoff: you buy a cheap front end with expensive indexing.
Use a `deque` for BFS and sliding windows; use a `list` for anything index-heavy.

`deque(maxlen=k)` auto-evicts from the far end — a free fixed-size rolling window,
genuinely useful in ETL code, not just interviews.

### Monotonic deque — sliding window maximum
The hard version of the sliding-window pattern. Hold indices whose values are
**decreasing**; the front is always the window's max. Pop from the back anything
smaller than the incoming value (it can never be a max again), pop from the front
anything that has slid out of the window. O(n).

In [ ]:
dq = deque([1, 2, 3])
dq.appendleft(0); dq.append(4)
assert list(dq) == [0, 1, 2, 3, 4]
assert dq.popleft() == 0 and dq.pop() == 4        # both O(1)

roll = deque(maxlen=3)                            # free rolling window
for x in [1, 2, 3, 4, 5]:
    roll.append(x)
assert list(roll) == [3, 4, 5]

N = 30_000
t = time.perf_counter(); q = deque(range(N))
while q: q.popleft()
d = time.perf_counter() - t
t = time.perf_counter(); l = list(range(N))
while l: l.pop(0)
li = time.perf_counter() - t
print(f"deque.popleft {d*1000:6.1f} ms | list.pop(0) {li*1000:7.1f} ms | {li/d:.0f}x")

In [ ]:
def max_sliding_window(nums, k):
    """Max of every window of size k. O(n) time, O(k) space.

    dq holds INDICES, values strictly decreasing -> dq[0] is the window max.
    """
    if not nums or k <= 0:
        return []
    dq, out = deque(), []
    for i, v in enumerate(nums):
        while dq and nums[dq[-1]] <= v:      # smaller predecessors are now useless
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                   # front slid out of the window
            dq.popleft()
        if i >= k - 1:                       # window is full -> record
            out.append(nums[dq[0]])
    return out

assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
assert max_sliding_window([9, 8, 7], 2) == [9, 8]     # decreasing input
assert max_sliding_window([], 3) == []
print("max_sliding_window passes -- O(n), not O(n*k)")

---
# 6. Linked list

Nodes holding a value and a `next` pointer. Not contiguous, so:

- **No random access.** Reaching index `i` costs O(i). No binary search, ever.
- **O(1) insert/delete** — *if you already hold the node before it.* That caveat
  is the whole value proposition.
- Extra memory per element (the pointer) and poor cache locality. In real Python
  code you almost never want one; in interviews they are pointer-manipulation
  exercises.

### The two techniques
1. **Dummy head.** Allocate a throwaway node before the real head so "insert at
   front" and "delete the head" stop being special cases. Return `dummy.next`.
   This removes most of the `if head is None` bugs.
2. **Fast/slow pointers** (Floyd). `slow` moves 1, `fast` moves 2.
   - `fast` hits the end → `slow` is at the **middle**.
   - `fast` meets `slow` → there is a **cycle**.
   - Start `fast` `k` ahead → gap stays `k` → `slow` lands on the **k-th from the end**.
   All O(1) space, which is usually the point of the question.

**Reversal is the one to have in muscle memory** — three pointers, and you must
save `nxt` *before* you overwrite `cur.next` or you drop the rest of the list.

In [ ]:
class Node:
    __slots__ = ("val", "next")           # __slots__: no per-node __dict__
    def __init__(self, val, nxt=None):
        self.val, self.next = val, nxt

def build(values):
    """List -> linked list, using a dummy head so there's no special first case."""
    dummy = tail = Node(None)
    for v in values:
        tail.next = Node(v)
        tail = tail.next
    return dummy.next

def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

assert to_list(build([1, 2, 3])) == [1, 2, 3]
assert to_list(build([])) == []

def reverse(head):
    """Iterative reversal. O(n) time, O(1) space. Know this cold."""
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1. SAVE the rest of the list first
        cur.next = prev     # 2. flip this node's pointer backwards
        prev = cur          # 3. advance prev
        cur = nxt           # 4. advance cur
    return prev             # cur is None; prev is the new head

assert to_list(reverse(build([1, 2, 3, 4]))) == [4, 3, 2, 1]
assert to_list(reverse(build([1]))) == [1]
assert reverse(build([])) is None
print("build / reverse pass")

In [ ]:
def middle(head):
    """Fast/slow: when fast reaches the end, slow is halfway. O(1) space."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow

assert middle(build([1, 2, 3, 4, 5])).val == 3        # odd -> exact middle
assert middle(build([1, 2, 3, 4])).val == 3           # even -> second of the two

def has_cycle(head):
    """Floyd: in a cycle the fast pointer gains 1 step per iteration, so it
    must eventually land exactly on slow. O(n) time, O(1) space."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

straight = build([1, 2, 3, 4])
assert not has_cycle(straight)
looped = build([1, 2, 3, 4])
tail = looped
while tail.next: tail = tail.next
tail.next = looped.next                                # 4 -> 2, cycle
assert has_cycle(looped)

def nth_from_end(head, n):
    """Open a gap of n, then walk both. One pass, O(1) space."""
    dummy = Node(None, head)
    fast = slow = dummy
    for _ in range(n):
        if not fast.next:
            return None                                # n longer than the list
        fast = fast.next
    while fast.next:
        fast, slow = fast.next, slow.next
    return slow.next

assert nth_from_end(build([1, 2, 3, 4, 5]), 2).val == 4
assert nth_from_end(build([1, 2]), 5) is None
print("fast/slow trio passes")

---
# 7. Heap / priority queue — `heapq`

A **complete binary tree stored in a flat array**. No pointers — the tree shape is
pure index arithmetic:

```
parent(i) = (i - 1) // 2      left(i) = 2i + 1      right(i) = 2i + 2
```

**Heap invariant:** every parent ≤ both children (a *min*-heap). So `h[0]` is the
minimum. Note what this does **not** give you: the heap is *not* sorted, and
`h[1] <= h[2]` is not guaranteed. Only the root is special.

`push`/`pop` restore the invariant by sifting an element up or down one level at a
time — the tree is O(log n) deep, so both are **O(log n)**. Peeking `h[0]` is O(1).

`heapq.heapify(lst)` is **O(n)**, not O(n log n) — it sifts down from the last
parent, and most nodes are near the bottom where sifting is cheap. Knowing this
is a real signal.

### Python-specific facts you must state
- **`heapq` is min-heap only.** For a max-heap, push `-x` (numbers) or negate the
  sort key. There is no `max-heap` flag.
- Tuples compare element-wise, so push `(priority, item)`. If `item` is not
  comparable (e.g. a dict), add a tiebreaker counter: `(priority, count, item)` —
  otherwise a priority tie raises `TypeError`.
- `heapq.nlargest(k, it)` / `nsmallest` exist and are fine to use.

### Top-K: the size-K heap trick
For the K **largest**, keep a **min**-heap of size K. The smallest of your K best
sits at the root, so it is O(1) to test and O(log K) to evict.
→ **O(n log K)** time, **O(K)** space — beats sorting's O(n log n), and it streams.

In [ ]:
def sift_down(h, i, n):
    """Push h[i] down until the heap invariant holds below it."""
    while True:
        smallest, l, r = i, 2 * i + 1, 2 * i + 2
        if l < n and h[l] < h[smallest]: smallest = l
        if r < n and h[r] < h[smallest]: smallest = r
        if smallest == i:
            return
        h[i], h[smallest] = h[smallest], h[i]
        i = smallest

def my_heapify(lst):
    """O(n), not O(n log n): start at the last parent and sift down."""
    for i in range(len(lst) // 2 - 1, -1, -1):
        sift_down(lst, i, len(lst))
    return lst

def my_heappop(h):
    top = h[0]
    h[0] = h[-1]              # move the last leaf to the root
    h.pop()
    if h:
        sift_down(h, 0, len(h))
    return top

data = [5, 3, 8, 1, 9, 2, 7]
mine = my_heapify(data[:])
assert mine[0] == 1                                   # root is the minimum
drained = [my_heappop(mine) for _ in range(len(data))]
assert drained == sorted(data)                         # that's heapsort

ref = data[:]; heapq.heapify(ref)
assert [heapq.heappop(ref) for _ in range(len(ref))] == sorted(data)
print("from-scratch heap agrees with heapq:", drained)

In [ ]:
def top_k_largest(nums, k):
    """K largest via a size-K MIN-heap. O(n log k) time, O(k) space.

    The root is the weakest of the current K best -> O(1) to compare against.
    """
    if k <= 0:
        return []
    h = []
    for v in nums:
        if len(h) < k:
            heapq.heappush(h, v)
        elif v > h[0]:                 # better than our current worst-of-best
            heapq.heapreplace(h, v)    # pop+push in one sift -> faster than two ops
    return sorted(h, reverse=True)

assert top_k_largest([3, 1, 5, 12, 2, 11], 3) == [12, 11, 5]
assert top_k_largest([1, 2], 5) == [2, 1]              # k > n
assert top_k_largest([1, 2, 3], 0) == []

def merge_k_sorted(lists):
    """K-way merge. O(n log k) -- the heap only ever holds one item per list."""
    h = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]   # (val, list#, idx)
    heapq.heapify(h)
    out = []
    while h:
        val, li, idx = heapq.heappop(h)
        out.append(val)
        if idx + 1 < len(lists[li]):                 # refill from the same list
            heapq.heappush(h, (lists[li][idx + 1], li, idx + 1))
    return out

assert merge_k_sorted([[1, 4, 5], [1, 3, 4], [2, 6]]) == [1, 1, 2, 3, 4, 4, 5, 6]
assert merge_k_sorted([[], [1], []]) == [1]
assert merge_k_sorted([]) == []

# The tuple-comparison trap, made concrete:
try:
    heapq.heappush([], (1, {"a": 1})); heapq.heappush([(1, {"a": 1})], (1, {"b": 2}))
except TypeError as e:
    print("priority tie on an uncomparable payload ->", e, "-> add a counter")
print("top-K + k-way merge pass")

In [ ]:
class MedianFinder:
    """Streaming median with TWO heaps -- your educative/6_Two_Heaps pattern.

    low  = max-heap (negated) of the smaller half
    high = min-heap            of the larger half
    Invariants: every low <= every high, and len(low) - len(high) in {0, 1}.
    add O(log n), median O(1).
    """
    def __init__(self):
        self.low, self.high = [], []

    def add(self, num):
        heapq.heappush(self.low, -num)                       # always via low
        heapq.heappush(self.high, -heapq.heappop(self.low))  # pass its max over
        if len(self.high) > len(self.low):                   # rebalance
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if not self.low:
            return None
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2

mf = MedianFinder()
for x, expected in [(1, 1.0), (2, 1.5), (3, 2.0), (4, 2.5), (5, 3.0)]:
    mf.add(x)
    assert mf.median() == expected, (x, mf.median())
mf2 = MedianFinder()
for x in [5, 4, 3, 2, 1]:                 # reverse order still works
    mf2.add(x)
assert mf2.median() == 3.0
print("MedianFinder passes")

---
# 8. Binary tree & BST

A tree is a graph with no cycles and one parent per node. Almost every tree
question is **"which traversal order?"**

### The four traversals
| Order | Visit | Use it for |
|---|---|---|
| **Pre**-order | node → left → right | copying/serialising a tree |
| **In**-order | left → node → right | **a BST in sorted order** ← the big one |
| **Post**-order | left → right → node | anything needing children's answers first (height, delete, bottom-up DP) |
| **Level**-order (BFS) | row by row, `deque` | "by depth", shortest path, right-side view |

### BST invariant
`all(left subtree) < node < all(right subtree)` — **for the whole subtree**, not
just the immediate children. That is the classic "validate BST" bug: comparing
only `node.left.val < node.val` passes broken trees. Carry a `(lo, hi)` range down.

Search/insert/delete are O(h) where `h` is the height: **O(log n) balanced,
O(n) degenerate** (a sorted insertion order makes a linked list). Say "O(h)" and
then say what `h` is — that is the precise answer.

### Complexity
Any full traversal is **O(n) time**. Space is **O(h)** for the recursion stack
(DFS) or **O(w)** for the widest level (BFS) — and for a balanced tree
`w` ≈ n/2, so BFS can cost O(n) space where DFS costs O(log n).

In [ ]:
class TreeNode:
    __slots__ = ("val", "left", "right")
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

#         8
#      4     12
#     2 6   10 14
root = TreeNode(8,
    TreeNode(4, TreeNode(2), TreeNode(6)),
    TreeNode(12, TreeNode(10), TreeNode(14)))

def preorder(n):  return [] if not n else [n.val] + preorder(n.left) + preorder(n.right)
def inorder(n):   return [] if not n else inorder(n.left) + [n.val] + inorder(n.right)
def postorder(n): return [] if not n else postorder(n.left) + postorder(n.right) + [n.val]

assert preorder(root)  == [8, 4, 2, 6, 12, 10, 14]
assert inorder(root)   == [2, 4, 6, 8, 10, 12, 14]      # sorted <- it's a BST
assert postorder(root) == [2, 6, 4, 10, 14, 12, 8]

def level_order(n):
    """BFS. Returns a list per depth -- the shape most 'by level' questions want."""
    if not n:
        return []
    out, q = [], deque([n])
    while q:
        level = []
        for _ in range(len(q)):             # freeze the width BEFORE adding children
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        out.append(level)
    return out

assert level_order(root) == [[8], [4, 12], [2, 6, 10, 14]]
assert level_order(None) == []

def inorder_iterative(n):
    """No recursion -> no stack-overflow risk on a 10^5-deep degenerate tree."""
    out, stack, cur = [], [], n
    while cur or stack:
        while cur:                          # dive left, remembering the path
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        out.append(cur.val)
        cur = cur.right
    return out

assert inorder_iterative(root) == inorder(root)
print("all four traversals pass")

In [ ]:
def is_valid_bst(node, lo=float("-inf"), hi=float("inf")):
    """Validate a BST by narrowing an allowed RANGE, not by comparing neighbours."""
    if not node:
        return True
    if not (lo < node.val < hi):
        return False
    return (is_valid_bst(node.left,  lo, node.val) and
            is_valid_bst(node.right, node.val, hi))

assert is_valid_bst(root)

# The tree that defeats the naive 'check my children only' version:
#      5
#    1    7        <- 7 > 5 ok, and 4 < 7 ok... but 4 is in 5's RIGHT subtree
#        4 8
sneaky = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))
assert not is_valid_bst(sneaky)
def naive_bst(n):
    if not n: return True
    if n.left and n.left.val >= n.val: return False
    if n.right and n.right.val <= n.val: return False
    return naive_bst(n.left) and naive_bst(n.right)
assert naive_bst(sneaky)        # the bug, proven
print("range-based validation catches what neighbour-comparison misses")

def height(n):
    """Post-order: you need both children's answers before your own."""
    return 0 if not n else 1 + max(height(n.left), height(n.right))
assert height(root) == 3 and height(None) == 0

def lowest_common_ancestor(n, p, q):
    """LCA in a BST: walk down: both smaller -> left, both bigger -> right,
    otherwise they split here and THIS is the answer. O(h)."""
    while n:
        if p < n.val and q < n.val:   n = n.left
        elif p > n.val and q > n.val: n = n.right
        else:                         return n.val
    return None

assert lowest_common_ancestor(root, 2, 6) == 4
assert lowest_common_ancestor(root, 2, 14) == 8
assert lowest_common_ancestor(root, 4, 6) == 4      # p is itself the ancestor
print("is_valid_bst / height / LCA pass")

---
# 9. Trie (prefix tree)

A tree where **the path spells the key**. Each node holds a dict of children plus
an `is_word` flag. The root is the empty prefix.

- `insert` / `search` / `starts_with` → **O(L)** where L = length of the word.
  Crucially that is **independent of how many words are stored**.
- A `set` also gives O(L) exact lookup (hashing reads the whole key) — so a trie is
  only worth it when you need **prefix** operations: autocomplete, "all words with
  this prefix", wildcard matching, longest common prefix.
- Cost: lots of small nodes. Memory is the tradeoff.

If the question says *prefix*, *autocomplete*, or *starts with*, say "trie" —
the alternative is scanning every word, O(n·L).

In [ ]:
class Trie:
    def __init__(self):
        self.children = {}          # char -> Trie
        self.is_word = False

    def insert(self, word):
        node = self
        for ch in word:
            node = node.children.setdefault(ch, Trie())   # walk, creating as needed

        node.is_word = True

    def _walk(self, prefix):
        """Follow prefix; return the node there, or None."""
        node = self
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_word     # must be a terminal word

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

    def autocomplete(self, prefix):
        """Every stored word beginning with prefix, sorted."""
        node = self._walk(prefix)
        if node is None:
            return []
        out = []
        def dfs(n, path):
            if n.is_word:
                out.append(prefix + path)
            for ch, child in n.children.items():
                dfs(child, path + ch)
        dfs(node, "")
        return sorted(out)


t = Trie()
for w in ["car", "card", "care", "cat", "dog"]:
    t.insert(w)

assert t.search("car")                    # stored word
assert not t.search("ca")                 # a prefix is NOT a word
assert t.starts_with("ca")                # ...but it is a valid prefix
assert not t.starts_with("cz")
assert t.autocomplete("car") == ["car", "card", "care"]
assert t.autocomplete("d") == ["dog"]
assert t.autocomplete("zzz") == []
print("Trie passes -- prefix queries cost O(len), not O(n_words)")

---
# 10. Graph — BFS, DFS, topological sort

### Representation — always say adjacency list
| | space | "is u→v an edge?" | "neighbours of u" |
|---|---|---|---|
| **Adjacency list** `{u: [v, ...]}` | O(V+E) | O(deg u) | **O(deg u)** |
| Adjacency matrix | O(V²) | O(1) | O(V) |

Real graphs are **sparse**, and every traversal iterates neighbours, so the list
wins. Use a matrix only for dense graphs or when you constantly test single edges.

### BFS vs DFS — pick on purpose
- **BFS** (`deque`, pop **left**): explores by distance, so the **first time you
  reach a node is via a shortest path** — but only when every edge costs the same.
  Weighted edges need Dijkstra (a heap instead of a queue).
- **DFS** (recursion or an explicit stack): explores one branch to the end. Use for
  connectivity, cycle detection, all-paths, flood fill, topological order.

Both are **O(V + E)** time — every node once, every edge once.

### The non-negotiable rule
**Mark visited when you ENQUEUE, not when you dequeue.** Otherwise a node with
several in-edges gets pushed many times before you first pop it — the queue blows
up and complexity degrades. This is the #1 graph bug in interviews.

### Topological sort (Kahn)
Repeatedly emit a node with in-degree 0 and decrement its neighbours. If you emit
fewer than V nodes, the leftovers form a **cycle**. Free cycle detection.
Use it for prerequisites, build order, task dependencies — and note it out loud:
*a DAG is exactly the shape of an ETL DAG*, which is a nice bridge to your day job.

In [ ]:
def build_graph(edges, directed=False):
    g = defaultdict(list)
    for u, v in edges:
        g[u].append(v)
        if not directed:
            g[v].append(u)
    return g

#  1 -- 2 -- 4
#  |    |
#  3    5 -- 6          and an isolated 7
und = build_graph([(1, 2), (1, 3), (2, 4), (2, 5), (5, 6)])
und[7]          # touching a defaultdict CREATES the key -- here that's on purpose,
                # it registers node 7 as existing with no edges

def bfs_order(g, start):
    """Visit order, by increasing distance. O(V+E)."""
    seen = {start}                       # mark on ENQUEUE
    q, out = deque([start]), []
    while q:
        u = q.popleft()
        out.append(u)
        for v in g[u]:
            if v not in seen:
                seen.add(v)              # <-- here, NOT when popped
                q.append(v)
    return out

def shortest_path(g, start, goal):
    """Unweighted shortest path. BFS + a parent map to rebuild the route."""
    if start == goal:
        return [start]
    parent = {start: None}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in g[u]:
            if v in parent:
                continue
            parent[v] = u
            if v == goal:                       # reconstruct backwards
                path = [goal]
                while parent[path[-1]] is not None:
                    path.append(parent[path[-1]])
                return path[::-1]
            q.append(v)
    return []                                   # unreachable

assert bfs_order(und, 1) == [1, 2, 3, 4, 5, 6]
assert shortest_path(und, 1, 6) == [1, 2, 5, 6]
assert shortest_path(und, 1, 7) == []           # disconnected
assert shortest_path(und, 3, 3) == [3]

def dfs_iterative(g, start):
    """Explicit stack -> no recursion limit. Note: pop() from the RIGHT."""
    seen, stack, out = set(), [start], []
    while stack:
        u = stack.pop()
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
        for v in reversed(g[u]):          # reversed -> visit in natural order
            if v not in seen:
                stack.append(v)
    return out

assert dfs_iterative(und, 1) == [1, 2, 4, 5, 6, 3]

def connected_components(g, nodes):
    seen, count = set(), 0
    for n in nodes:
        if n not in seen:
            count += 1
            seen.update(bfs_order(g, n))
    return count

assert connected_components(und, [1, 2, 3, 4, 5, 6, 7]) == 2    # plus isolated 7
print("BFS / DFS / components pass")

In [ ]:
def topological_sort(n_nodes, edges):
    """Kahn's algorithm. Returns an order, or None if there's a cycle.

    Nodes are 0..n_nodes-1. edge (u, v) means 'u must come before v'.
    """
    g = defaultdict(list)
    indeg = [0] * n_nodes
    for u, v in edges:
        g[u].append(v)
        indeg[v] += 1

    q = deque(i for i in range(n_nodes) if indeg[i] == 0)   # nothing blocks these
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in g[u]:
            indeg[v] -= 1                  # one prerequisite satisfied
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == n_nodes else None          # short -> cycle

order = topological_sort(6, [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)])
assert order is not None and len(order) == 6
pos = {v: i for i, v in enumerate(order)}
for u, v in [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]:
    assert pos[u] < pos[v], (u, v)                  # every constraint respected
assert topological_sort(2, [(0, 1), (1, 0)]) is None            # 2-cycle
assert topological_sort(3, []) == [0, 1, 2]                     # no edges
print("topological_sort passes; order =", order)

def can_finish_courses(n, prereqs):
    """LeetCode 'Course Schedule' == 'does a topological order exist'."""
    return topological_sort(n, [(p, c) for c, p in prereqs]) is not None

assert can_finish_courses(2, [[1, 0]])
assert not can_finish_courses(2, [[1, 0], [0, 1]])
print("course schedule reduces to cycle detection")

In [ ]:
def num_islands(grid):
    """Grid problems ARE graph problems: each cell is a node, neighbours are the
    4 orthogonal cells. BFS flood fill. O(rows*cols)."""
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    seen, count = set(), 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "1" or (r, c) in seen:
                continue
            count += 1
            q = deque([(r, c)]); seen.add((r, c))
            while q:
                y, x = q.popleft()
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if (0 <= ny < rows and 0 <= nx < cols          # in bounds
                            and grid[ny][nx] == "1"                # is land
                            and (ny, nx) not in seen):             # unvisited
                        seen.add((ny, nx))
                        q.append((ny, nx))
    return count

assert num_islands([list("11000"), list("11000"), list("00100"), list("00011")]) == 3
assert num_islands([list("000")]) == 0
assert num_islands([]) == 0
print("num_islands passes -- a grid is just an implicit graph")

---
# 11. Union-Find (Disjoint Set Union)

Answers two questions on a collection of disjoint groups:
**"are these two in the same group?"** and **"merge these two groups."**

Each element points at a parent; the root identifies the group.

Two optimisations, and you need both:
1. **Path compression** — during `find`, re-point every node on the path straight
   at the root. Paths get flat.
2. **Union by rank/size** — always hang the smaller tree under the bigger one, so
   depth never grows unnecessarily.

Together: **O(α(n))** amortised — the inverse Ackermann function, below 5 for any
n you will ever see. Just say "effectively constant".

### Union-find vs BFS/DFS — the real distinction
Both find connected components. Choose union-find when edges **arrive
incrementally** and you must answer connectivity *along the way* — streaming
edges, Kruskal's MST, "count components after each merge", detecting a cycle in an
undirected graph as you add edges. If you have the whole static graph up front and
just want components once, BFS is simpler. Saying *why* you picked it matters more
than the implementation.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))       # every node starts as its own root
        self.size = [1] * n
        self.n_components = n

    def find(self, x):
        """Root of x, compressing the path on the way (iterative -> no recursion)."""
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:      # second pass: re-point everything
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        """Merge. Returns False if they were already together (-> that edge closes
        a cycle), which is how you detect cycles in an undirected graph."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:  # union by size: small under large
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.n_components -= 1
        return True

    def connected(self, a, b):
        return self.find(a) == self.find(b)


uf = UnionFind(10)
for a, b in [(0, 1), (1, 2), (3, 4), (5, 6), (6, 7), (7, 5)]:
    uf.union(a, b)
assert uf.connected(0, 2) and not uf.connected(0, 3)
assert uf.n_components == 5             # {0,1,2} {3,4} {5,6,7} {8} {9}
assert uf.union(0, 2) is False          # already merged -> would close a cycle
assert uf.find(0) == uf.find(1) == uf.find(2)

def has_cycle_undirected(n, edges):
    """An edge between two nodes already in the same set closes a cycle."""
    uf = UnionFind(n)
    return any(not uf.union(u, v) for u, v in edges)

assert not has_cycle_undirected(4, [(0, 1), (1, 2), (2, 3)])    # a path
assert has_cycle_undirected(3, [(0, 1), (1, 2), (2, 0)])        # a triangle

def count_components_streaming(n, edges):
    """The thing BFS can't do cheaply: the count AFTER EACH edge arrives."""
    uf, out = UnionFind(n), []
    for u, v in edges:
        uf.union(u, v)
        out.append(uf.n_components)
    return out

assert count_components_streaming(5, [(0, 1), (1, 2), (3, 4), (0, 2)]) == [4, 3, 2, 2]
print("UnionFind passes -- note the last edge changed nothing (already merged)")

---
# 12. DP table — the structure is the array

DP is not really a data structure; it is a **table you fill so no subproblem is
solved twice**. Two prerequisites:

1. **Overlapping subproblems** — the naive recursion recomputes the same call.
2. **Optimal substructure** — the answer is built from answers to smaller versions.

### The only procedure you need
1. **Define the state in one English sentence.** `dp[i]` = *the answer for the
   first i items.* If you cannot say this sentence, you are not ready to code.
2. **Write the recurrence** — how does `dp[i]` follow from earlier entries?
3. **Base cases.**
4. **Iteration order** — every entry you read must already be filled.
5. **Optional: shrink the space.** If `dp[i]` only reads `dp[i-1]`, keep two
   variables instead of an array → O(1) space. Mention this even if you don't do it.

### Top-down vs bottom-up
- **Top-down (memo)** — write the recursion, add `@lru_cache`. Fast to get right,
  risks a deep recursion stack.
- **Bottom-up (table)** — a loop. No stack risk, allows the O(1)-space trick.

Both are the same complexity. Write whichever you can get correct fastest, then
say you could convert it — that answer is always right.

In [ ]:
from functools import lru_cache

# One problem, three ways -- watch the state definition stay identical.
def stairs_naive(n):
    """Exponential: recomputes the same call over and over."""
    if n <= 2:
        return max(n, 1)
    return stairs_naive(n - 1) + stairs_naive(n - 2)

@lru_cache(maxsize=None)
def stairs_memo(n):
    """Top-down. O(n) time, O(n) space (dict + recursion stack)."""
    if n <= 2:
        return max(n, 1)
    return stairs_memo(n - 1) + stairs_memo(n - 2)

def stairs_table(n):
    """Bottom-up. dp[i] = number of ways to reach step i."""
    if n <= 2:
        return max(n, 1)
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

def stairs_o1(n):
    """Same recurrence, O(1) space -- dp[i] only ever reads the last two."""
    if n <= 2:
        return max(n, 1)
    a, b = 1, 2
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

for n in range(1, 16):
    assert stairs_naive(n) == stairs_memo(n) == stairs_table(n) == stairs_o1(n), n
assert stairs_o1(0) == 1 and stairs_o1(40) == 165_580_141

t = time.perf_counter(); stairs_naive(28); naive_t = time.perf_counter() - t
t = time.perf_counter(); stairs_o1(28);    dp_t = time.perf_counter() - t
print(f"n=28  naive {naive_t*1000:7.2f} ms | O(1)-space DP {dp_t*1000:0.4f} ms")

In [ ]:
def edit_distance(a, b):
    """Levenshtein. THE 2D DP template.

    dp[i][j] = min edits to turn a[:i] into b[:j].
    O(len(a) * len(b)) time and space.
    """
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i                 # delete every char of a
    for j in range(n + 1):
        dp[0][j] = j                 # insert every char of b
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]          # free: chars match
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],     # delete from a
                                   dp[i][j - 1],     # insert into a
                                   dp[i - 1][j - 1]) # substitute
    return dp[m][n]

assert edit_distance("horse", "ros") == 3
assert edit_distance("", "abc") == 3
assert edit_distance("abc", "abc") == 0
assert edit_distance("", "") == 0

def coin_change(coins, amount):
    """Fewest coins to make amount, or -1. dp[x] = fewest coins for x."""
    INF = float("inf")
    dp = [0] + [INF] * amount
    for x in range(1, amount + 1):
        for c in coins:
            if c <= x and dp[x - c] + 1 < dp[x]:
                dp[x] = dp[x - c] + 1
    return -1 if dp[amount] == INF else dp[amount]

assert coin_change([1, 2, 5], 11) == 3      # 5+5+1
assert coin_change([2], 3) == -1            # impossible
assert coin_change([1], 0) == 0
print("edit_distance + coin_change pass")

---
# 13. Complexity — the table, and what the curves look like

### Python built-ins, average case
| Operation | `list` | `deque` | `dict`/`set` | heap (`heapq`) |
|---|---|---|---|---|
| index `x[i]` | **O(1)** | O(n) | — | O(1) peek min |
| search `x in c` | O(n) | O(n) | **O(1)** | O(n) |
| insert at end | O(1)* | O(1) | **O(1)** | O(log n) |
| insert at front | O(n) | **O(1)** | — | — |
| delete from end | O(1) | O(1) | **O(1)** | O(log n) |
| delete from front | O(n) | **O(1)** | — | O(log n) pop min |
| min / max | O(n) | O(n) | O(n) | **O(1)** min |

\* amortised

### Structures
| Structure | Search | Insert | Delete | Space |
|---|---|---|---|---|
| Dynamic array | O(n) | O(1)* end / O(n) front | O(n) | O(n) |
| Hash map | O(1) avg, O(n) worst | O(1) avg | O(1) avg | O(n) |
| Linked list | O(n) | O(1) given the node | O(1) given the node | O(n) |
| Stack / queue | O(n) | O(1) | O(1) | O(n) |
| Heap | O(n) | O(log n) | O(log n) pop min | O(n) |
| Balanced BST | O(log n) | O(log n) | O(log n) | O(n) |
| Trie | O(L) | O(L) | O(L) | O(n·L) |
| Union-find | ~O(1) | ~O(1) union | — | O(n) |

### What the input size tells you
The constraint is a hint about the intended complexity. If n ≤ 20, an exponential
backtracking answer is *expected*; if n = 10⁵, O(n²) will time out and they want
O(n log n) or better.

| n | What fits |
|---|---|
| ≤ 20 | O(2ⁿ) — backtracking, subsets |
| ≤ 500 | O(n³) |
| ≤ 5,000 | O(n²) |
| ≤ 10⁶ | O(n log n) — sorting, heaps |
| > 10⁶ | O(n) or O(log n) only |

In [ ]:
# Growth curves. CP AXTRA palette; brand yellow and Lotus green are stepped
# slightly to pass the lightness/contrast checks. Every curve is directly
# labelled, so identity never depends on colour alone.
import matplotlib.pyplot as plt
import numpy as np

CPX_BLUE, CPX_YELLOW = "#306FC7", "#C8901F"      # CP AXTRA
LOTUS_GREEN, MAKRO_RED = "#0F8A83", "#DA3832"    # Lotus, Makro
INK, MUTED, GRID = "#1f2328", "#6b7280", "#e5e7eb"

n = np.linspace(1, 100, 400)
series = [
    ("O(1)",        np.ones_like(n),        MUTED),
    ("O(log n)",    np.log2(n),             LOTUS_GREEN),
    ("O(n)",        n,                      CPX_BLUE),
    ("O(n log n)",  n * np.log2(n),         CPX_YELLOW),
    ("O(n²)",  n ** 2,                 MAKRO_RED),
]

fig, ax = plt.subplots(figsize=(8.5, 5))
fig.patch.set_facecolor("#fcfcfb"); ax.set_facecolor("#fcfcfb")

for label, y, colour in series:
    ax.plot(n, y, color=colour, linewidth=2, solid_capstyle="round")
    ax.annotate(label, xy=(n[-1], y[-1]), xytext=(6, 0),          # direct label
                textcoords="offset points", color=colour,
                fontsize=10, fontweight="semibold", va="center")

ax.set_yscale("log")                    # without this, n^2 flattens everything else
ax.set_xlim(1, 100); ax.set_ylim(0.5, 2e4)
ax.set_xlabel("input size n", color=MUTED, fontsize=10)
ax.set_ylabel("operations (log scale)", color=MUTED, fontsize=10)
ax.set_title("Why complexity class decides whether you pass",
             color=INK, fontsize=13, fontweight="semibold", loc="left", pad=14)
ax.grid(True, axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
fig.subplots_adjust(right=0.86)         # room for the direct labels
plt.show()

# The same story as a table, for anyone who can't read the colours.
print(f"\n{'n':>8} {'log n':>8} {'n':>10} {'n log n':>12} {'n^2':>14}")
for size in (10, 100, 1_000, 10_000, 100_000):
    print(f"{size:>8} {np.log2(size):>8.1f} {size:>10,} "
          f"{size*np.log2(size):>12,.0f} {size**2:>14,}")

---
# 14. Self-test — no answers here

Same rules as `blind/practice-without-view.ipynb`: **scroll up for nothing.**
Set a timer, write the function, make the asserts pass.

Meta's screen gives you roughly **7 minutes per Python question**. Time yourself.
If you blow the budget, the fix is usually that you started coding before naming
the structure — say the structure out loud first, then type.

For each one, before writing code, answer in a comment:
1. Which structure, and why?
2. Time and space complexity?
3. What are the edge cases?

Then implement. The asserts are the spec.

In [ ]:
# DRILL 1 -- 5 min.  Structure? Complexity?
def first_unique_char(s):
    """Index of the first non-repeating character, or -1."""
    pass

assert first_unique_char("leetcode") == 0
assert first_unique_char("loveleetcode") == 2
assert first_unique_char("aabb") == -1
assert first_unique_char("") == -1
print("drill 1 ok")

In [ ]:
# DRILL 2 -- 7 min.  Two structures here, one for counting, one for the top-K.
def top_k_frequent(nums, k):
    """The k most frequent values, most frequent first.
    Ties: smaller value first. Aim for O(n log k), not O(n log n)."""
    pass

assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]
assert top_k_frequent([1], 1) == [1]
assert top_k_frequent([4, 4, 5, 5], 2) == [4, 5]      # tie -> smaller first
assert top_k_frequent([], 3) == []
print("drill 2 ok")

In [ ]:
# DRILL 3 -- 7 min.  Your blind/4 and blind/10 problems are this one.
def max_concurrent_meetings(intervals):
    """Max number of meetings overlapping at any instant.
    [start, end) -- a meeting ending at 10 does NOT overlap one starting at 10.
    Hint: two structures work. A heap of end-times, or sort the +1/-1 events."""
    pass

assert max_concurrent_meetings([(1, 4), (2, 5), (7, 9)]) == 2
#           at t=11 these four are live: (3,19) (8,12) (10,20) (11,15)
assert max_concurrent_meetings([(1, 10), (2, 7), (3, 19), (8, 12), (10, 20), (11, 15)]) == 4
assert max_concurrent_meetings([(1, 2), (2, 3), (3, 4)]) == 1     # touching, not overlapping
assert max_concurrent_meetings([]) == 0
print("drill 3 ok")

In [ ]:
# DRILL 4 -- 7 min.  Which traversal?
def right_side_view(root):
    """Values visible from the right: the last node of each level, top to bottom.
    root is a TreeNode from section 8 (or None)."""
    pass

#      1
#    2   3
#     5   4
t = TreeNode(1, TreeNode(2, None, TreeNode(5)), TreeNode(3, None, TreeNode(4)))
assert right_side_view(t) == [1, 3, 4]
assert right_side_view(TreeNode(1)) == [1]
assert right_side_view(None) == []
print("drill 4 ok")

In [ ]:
# DRILL 5 -- 8 min.  Graph. Mark visited WHERE?
def word_ladder_length(begin, end, word_list):
    """Shortest transformation chain begin -> end, changing one letter at a time,
    every intermediate word in word_list. Return the number of words in the
    chain, or 0 if impossible. (begin need not be in word_list; end must be.)"""
    pass

assert word_ladder_length("hit", "cog", ["hot","dot","dog","lot","log","cog"]) == 5
assert word_ladder_length("hit", "cog", ["hot","dot","dog","lot","log"]) == 0
assert word_ladder_length("a", "c", ["a","b","c"]) == 2
print("drill 5 ok")

In [ ]:
# DRILL 6 -- 8 min.  Sessionization -- your day job as an algorithm question.
def sessionize(events, gap_minutes=30):
    """events: list of (user_id, timestamp_minutes), unsorted.
    A new session starts when the gap from the previous event of the SAME user
    exceeds gap_minutes. Return {user_id: session_count}."""
    pass

evts = [("u1", 0), ("u1", 10), ("u1", 100), ("u2", 5), ("u1", 105), ("u2", 200)]
assert sessionize(evts) == {"u1": 2, "u2": 2}
assert sessionize([("u1", 0), ("u1", 30)]) == {"u1": 1}      # exactly 30 = same session
assert sessionize([("u1", 0), ("u1", 31)]) == {"u1": 2}
assert sessionize([]) == {}
print("drill 6 ok")

In [ ]:
# DRILL 7 -- 8 min.  DP. Write the state definition as a sentence FIRST.
def longest_increasing_subsequence(nums):
    """Length of the longest strictly increasing subsequence (not contiguous).
    O(n^2) DP is acceptable; O(n log n) with binary search is the strong answer."""
    pass

assert longest_increasing_subsequence([10, 9, 2, 5, 3, 7, 101, 18]) == 4   # 2,3,7,101
assert longest_increasing_subsequence([7, 7, 7]) == 1
assert longest_increasing_subsequence([]) == 0
print("drill 7 ok")

In [ ]:
# DRILL 8 -- 6 min.  Union-find or DFS? Justify your pick in a comment.
def accounts_merge_count(accounts):
    """accounts: list of [name, email1, email2, ...].
    Two accounts belong to the same person if they share ANY email.
    Return the number of distinct people."""
    pass

assert accounts_merge_count([
    ["John", "a@x.com", "b@x.com"],
    ["John", "b@x.com", "c@x.com"],     # shares b -> same person
    ["Mary", "m@x.com"],
    ["John", "z@x.com"],                # same NAME, no shared email -> different
]) == 3
assert accounts_merge_count([]) == 0
print("drill 8 ok")

---
## After the drills

Score yourself honestly on **time**, not just correctness — the `blind/` README is
right that the clock is the enemy, and the 45-minute screen has no room for a
restart.

When a drill goes badly, the diagnosis is almost always one of three things:

1. **Wrong structure.** You reached for a list where a dict/heap/deque was the
   answer. Fix: re-read the decision table at the top until symptom → structure is
   reflex.
2. **Right structure, fumbled API.** You knew "heap" but not `heapreplace`, or
   "counter" but not `most_common`. Fix: re-run sections 3, 5 and 7 and type the
   built-ins from memory.
3. **Missed an edge case.** Empty input, single element, all-equal, k > n, ties.
   Fix: before coding, write the edge-case list as a comment — the habit costs 20
   seconds and saves the question.

Then go back to `educative/` for volume on whichever pattern broke.